# TabPFN-TS with Covariates

**Key insight for mechanical systems**: Sensor readings depend on operating conditions!

Examples:
- Vibration amplitude depends on motor **speed** (rpm)
- Hydraulic pressure depends on **load** setting
- Temperature depends on **ambient conditions** and duty cycle

TabPFN-TS natively supports covariates—external variables that provide context for the forecast.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

np.random.seed(42)

## 1. Create Covariate-Dependent Data

Simulate a machine where vibration depends on speed setting.

In [ ]:
# Simulate 300 timesteps
n_samples = 300
t = np.arange(n_samples)

# Operating condition: speed varies over time
# Start slow, ramp up, stay high, ramp down
speed = np.concatenate([
    np.linspace(500, 500, 50),    # Low speed
    np.linspace(500, 1500, 50),   # Ramp up
    np.linspace(1500, 1500, 100), # High speed
    np.linspace(1500, 800, 50),   # Ramp down
    np.linspace(800, 800, 50),    # Medium speed
])

# Vibration depends on speed (simplified physics)
# Higher speed → higher vibration amplitude
base_vibration = 0.5 * np.sin(2 * np.pi * 0.1 * t)  # Periodic component
speed_effect = 0.0005 * speed  # Speed-dependent amplitude
noise = 0.05 * np.random.randn(n_samples)

vibration = speed_effect * (1 + base_vibration) + noise

print(f"Speed range: {speed.min():.0f} - {speed.max():.0f} rpm")
print(f"Vibration range: {vibration.min():.3f} - {vibration.max():.3f} mm/s")

In [ ]:
# Visualize the relationship
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, speed, 'b-', linewidth=1)
axes[0].set_ylabel('Speed (rpm)')
axes[0].set_title('Operating Condition: Motor Speed')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, vibration, 'r-', linewidth=0.8)
axes[1].set_ylabel('Vibration (mm/s)')
axes[1].set_xlabel('Time (samples)')
axes[1].set_title('Sensor Reading: Vibration (depends on speed)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Forecast Without Covariates (Baseline)

In [ ]:
from tabpfn_ts import TabPFNForecaster

# Train/test split
train_size = 200
y_train = vibration[:train_size]
y_test = vibration[train_size:]
speed_train = speed[:train_size]
speed_test = speed[train_size:]
horizon = len(y_test)

print(f"Training: {train_size} samples")
print(f"Test: {horizon} samples")

In [ ]:
# Forecast WITHOUT covariates
forecaster_no_cov = TabPFNForecaster(horizon=horizon)
forecaster_no_cov.fit(y_train)
pred_no_cov = forecaster_no_cov.predict()

rmse_no_cov = np.sqrt(mean_squared_error(y_test, pred_no_cov))
print(f"RMSE without covariates: {rmse_no_cov:.4f}")

## 3. Forecast WITH Covariates

In [ ]:
# Prepare covariates
# Note: We need future covariates for the forecast horizon
# In real applications, these might be planned operating conditions

X_train = speed_train.reshape(-1, 1)  # Shape: (n_train, n_features)
X_test = speed_test.reshape(-1, 1)    # Known future operating conditions

print(f"Covariate training shape: {X_train.shape}")
print(f"Covariate test shape: {X_test.shape}")

In [ ]:
# Forecast WITH covariates
forecaster_with_cov = TabPFNForecaster(horizon=horizon)

# Fit with historical data AND covariate values
forecaster_with_cov.fit(y_train, X=X_train)

# Predict using known future covariates
pred_with_cov = forecaster_with_cov.predict(X=X_test)

rmse_with_cov = np.sqrt(mean_squared_error(y_test, pred_with_cov))
print(f"RMSE with covariates: {rmse_with_cov:.4f}")
print(f"Improvement: {100*(rmse_no_cov - rmse_with_cov)/rmse_no_cov:.1f}%")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Speed (covariate)
axes[0].plot(t, speed, 'b-', linewidth=1)
axes[0].axvline(x=train_size, color='gray', linestyle=':', alpha=0.7)
axes[0].set_ylabel('Speed (rpm)')
axes[0].set_title('Operating Condition (Covariate)')
axes[0].grid(True, alpha=0.3)

# Vibration forecasts
axes[1].plot(t[:train_size], y_train, 'b-', label='Training', linewidth=0.8)
axes[1].plot(t[train_size:], y_test, 'g-', label='True', linewidth=1.5)
axes[1].plot(t[train_size:], pred_no_cov, 'r--', label=f'No covariates (RMSE={rmse_no_cov:.3f})', linewidth=1.5)
axes[1].plot(t[train_size:], pred_with_cov, 'm--', label=f'With covariates (RMSE={rmse_with_cov:.3f})', linewidth=1.5)
axes[1].axvline(x=train_size, color='gray', linestyle=':', alpha=0.7)
axes[1].set_ylabel('Vibration (mm/s)')
axes[1].set_xlabel('Time (samples)')
axes[1].set_title('Forecast Comparison: With vs Without Covariates')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Multiple Covariates

Mechanical systems often have multiple operating conditions:
- Speed (rpm)
- Load (N or %)
- Ambient temperature (°C)

In [ ]:
# Add more covariates
load = 50 + 30 * np.sin(2 * np.pi * 0.02 * t)  # Cyclic load
ambient_temp = 25 + 5 * np.sin(2 * np.pi * 0.005 * t)  # Slow temperature variation

# More realistic vibration model
vibration_multi = (
    0.0003 * speed +           # Speed effect
    0.002 * load +             # Load effect
    0.01 * ambient_temp +      # Temperature effect
    0.2 * np.sin(2 * np.pi * 0.1 * t) +  # Periodic
    0.05 * np.random.randn(n_samples)    # Noise
)

# Combine covariates
X_multi = np.column_stack([speed, load, ambient_temp])
print(f"Multi-covariate shape: {X_multi.shape}")
print(f"Covariates: speed, load, ambient_temp")

In [ ]:
# Compare forecasts with different covariate sets
y_train_m = vibration_multi[:train_size]
y_test_m = vibration_multi[train_size:]

results = {}

# No covariates
f1 = TabPFNForecaster(horizon=horizon)
f1.fit(y_train_m)
results['No covariates'] = np.sqrt(mean_squared_error(y_test_m, f1.predict()))

# Speed only
f2 = TabPFNForecaster(horizon=horizon)
f2.fit(y_train_m, X=speed[:train_size].reshape(-1, 1))
results['Speed only'] = np.sqrt(mean_squared_error(y_test_m, f2.predict(X=speed[train_size:].reshape(-1, 1))))

# All covariates
f3 = TabPFNForecaster(horizon=horizon)
f3.fit(y_train_m, X=X_multi[:train_size])
results['All covariates'] = np.sqrt(mean_squared_error(y_test_m, f3.predict(X=X_multi[train_size:])))

# Display results
print("\nRMSE Comparison:")
print("-" * 30)
for name, rmse in results.items():
    print(f"{name:20s}: {rmse:.4f}")

## 5. Relevance to Mechanical Systems

| Dataset | Available Covariates | Potential Use |
|---------|---------------------|---------------|
| **Hydraulic System** | Pressure settings, valve position | Predict flow/temperature from control inputs |
| **C-MAPSS** | Operating settings (3 values) | Predict sensor readings given flight conditions |
| **AURSAD** | Joint commands, velocities | Predict current/torque from motion commands |
| **Paderborn Bearing** | Load, speed (if recorded) | Predict vibration from operating conditions |

## 6. Exercises

### Exercise 1: Irrelevant Covariates

What happens if you add covariates that don't actually affect the target?

In [ ]:
# TODO: Add random noise as a "covariate" and check if it hurts performance
# noise_cov = np.random.randn(n_samples).reshape(-1, 1)
# ...

### Exercise 2: Cross-Sensor Conditioning

Use one sensor to help predict another (e.g., pressure → temperature)

In [ ]:
# TODO: Create two correlated sensors (e.g., pressure causes temperature rise)
# Use one as covariate to predict the other
# ...

### Exercise 3: Missing Future Covariates

In practice, we might not know future operating conditions. What happens?

In [ ]:
# TODO: Try forecasting with covariates, but use estimated/naive future covariates
# e.g., assume speed stays constant at last known value
# Compare to using true future covariates
# ...

## Summary

Key insights:

1. **Covariates can significantly improve forecasts** when the target depends on operating conditions
2. **TabPFN-TS handles covariates natively** — just pass them to `fit()` and `predict()`
3. **Future covariates must be known** — this is realistic for planned operations (scheduled speed profiles, known setpoints)
4. **Mechanical systems are a good fit** — they have clear input-output relationships (control → sensor)

**Next step:** `04_mechanical.ipynb` — Apply to real mechanical system data